# Re-analysis plan — Wang & Horowitz (2026), *Sci Rep* 16:11176

*Dogs (Canis familiaris) distinguish conspecific emotional chemosignals*

This notebook is the analysis I would run on this experiment. It is a **plan expressed
as working code**, not a replication: the deposited data (Dryad
`doi:10.5061/dryad.bzkh189pt`) sits behind an OAuth bearer token and returned HTTP 401,
so the notebook executes against synthetic fixtures built to the structure the Methods
section describes. **Every number produced below is a property of the simulator in
`simulate.py`, not a finding about dogs.** Point `COLUMN_MAP` at the real workbooks and
the same cells produce the real result.

### What this changes relative to the published analysis

| Published | Here | Why |
|---|---|---|
| Part 2 tested on 12 averaged time bins per condition | Dog-level summaries (9/9/6), plus a binomial GLMM with `(1\|dog)` on the bin-level binary data | Two reported H values (23.98, 23.11) exceed the maximum H attainable with 9/9/6 dogs (20.25); the tests were run on bins, so between-dog variance never entered the error term |
| Pooled Part 1 Wilcoxons over ~122 trials from 43 dogs | One mixed model with dog as a random intercept | ~3x pseudoreplication |
| Paired Wilcoxon on A4 vs B | Beta regression on the preference index B/(A4+B) | A4 and B are measured simultaneously under a near-fixed time budget, so they are negatively coupled by construction |
| ~36 p-values, no correction | BH-FDR across a pre-declared confirmatory family; everything else labelled exploratory | Positively dependent tests, discovery goal |
| V and p only | Hodges-Lehmann shift with bootstrap CI, rank-biserial r, epsilon-squared | A Wilcoxon V is not an effect size |
| Normality pre-test chooses ANOVA vs Kruskal-Wallis | Family fixed a priori by data type; permutation reference distributions | Pre-test-then-choose invalidates the nominal alpha, and Shapiro-Wilk at n=12 has almost no power |
| Gaussian GLM on counts out of 12 | Binomial GLMM; multinomial for the posture triple | Gaussian can predict outside the support and assumes constant variance |
| No reliability statistic | ICC(2,1) for durations, Cohen's kappa per ethogram item | Both primary outcomes are coder judgements |
| Nulls read as absence | Sensitivity analysis stating the smallest detectable effect | Part 2 has 80% power only for very large effects |

### What code cannot fix

Three problems are in the design, not the analysis, and no re-analysis reaches them:
a single odour donor (n = 1 at the stimulus level, so a species-level claim has no
error term); emotion fully confounded with physical exertion, restraint and fixed
collection order; and period, testing room and the owner-distraction task being
completely aliased. The notebook surfaces the third as a covariate it cannot
disentangle, and refuses to fit a donor random effect.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(pathlib.Path.cwd()))
import stats_helpers as H
import models as M
import simulate as S

pd.set_option("display.width", 120)
sns.set_theme(style="ticks", context="notebook")
OUT = pathlib.Path("../outputs"); OUT.mkdir(exist_ok=True)

## 1. Data contract

`load_deposit` is the only function that touches files. The column names in
`COLUMN_MAP` are inferred from the Methods, not read off the deposit — correct them
once `Part_1.xlsx` and `Part_2.xlsx` are in hand, and nothing downstream changes.

In [ ]:
COLUMN_MAP = {
    "part1": {
        "dog": "subject", "pairing": "odor_pairing",
        "presentation": "presentation", "investigation_s": "investigation_time_s",
        "side": "novel_side", "owner_enhanced": "owner_enhanced",
        "period": "collection_period",
    },
    "part2": {
        "dog": "subject", "condition": "odor_condition", "sex": "sex",
        "bin": "sample_point", "near_owner": "proximity_owner",
        "near_stranger": "proximity_stranger",
        "stress_behaviour": "stress_behavior", "joy_behaviour": "joy_behavior",
        "posture": "posture",
    },
}

CONFIRMATORY = [
    # pre-declared family: three pairing-level discrimination contrasts and
    # three pairing-level dishabituation contrasts. Everything else is exploratory.
    "discrimination:joy_baseline", "discrimination:stress_baseline",
    "discrimination:joy_stress", "habituation:joy_baseline",
    "habituation:stress_baseline", "habituation:joy_stress",
]


def load_deposit(path=None, use_synthetic=True):
    """Load Part 1 and Part 2, renaming deposit columns to canonical names.

    Falls back to synthetic fixtures so the pipeline is runnable and testable
    before the deposit is accessible. The return value carries a flag so no
    downstream output can be mistaken for a real result.
    """
    if use_synthetic or path is None:
        part1, part2 = S.simulate_part1(), S.simulate_part2()
        return {"part1": part1, "part2": part2, "synthetic": True}
    path = pathlib.Path(path)
    part1 = pd.read_excel(path / "Part_1.xlsx")
    part2 = pd.read_excel(path / "Part_2.xlsx")
    inv = {v: k for k, v in COLUMN_MAP["part1"].items()}
    part1 = part1.rename(columns=inv)
    inv2 = {v: k for k, v in COLUMN_MAP["part2"].items()}
    part2 = part2.rename(columns=inv2)
    return {"part1": part1, "part2": part2, "synthetic": False}


data = load_deposit()
part1, part2 = data["part1"], data["part2"]
print("SYNTHETIC FIXTURES" if data["synthetic"] else "REAL DEPOSIT")
print(f"Part 1: {part1.shape[0]} trials, {part1.dog.nunique()} dogs")
print(f"Part 2: {part2.shape[0]} sample points, {part2.dog.nunique()} dogs")
print(part2.groupby("condition").dog.nunique().to_dict())

## 2. Distributions before any test

Per the project's statistics framework, the first step is to look at the shapes rather
than to infer them from a normality test. The outcome is a floor-bounded, right-skewed
duration, which fixes the model family on its own.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].hist(part1.investigation_s, bins=30, color="#4C72B0")
axes[0].set_xlabel("investigation time (s)"); axes[0].set_ylabel("trials")
axes[1].hist(np.log1p(part1.investigation_s), bins=30, color="#55A868")
axes[1].set_xlabel("log1p(investigation time)")
from scipy import stats as st
st.probplot(np.log1p(part1.investigation_s), plot=axes[2])
axes[2].set_title("")
for ax in axes: sns.despine(ax=ax)
fig.suptitle("Part 1 outcome: raw, log1p, and QQ of log1p  [SYNTHETIC DATA]", y=1.04)
fig.tight_layout(); fig.savefig(OUT / "part1_distributions.png", dpi=200, bbox_inches="tight")

print(f"skew raw      {st.skew(part1.investigation_s):.2f}")
print(f"skew log1p    {st.skew(np.log1p(part1.investigation_s)):.2f}")
print(f"zeros         {(part1.investigation_s == 0).sum()}")

## 3. Part 1 — habituation and discrimination at the dog level

Two analyses, in order of evidential weight.

**3a. Dog-level paired contrasts with permutation reference distributions.** Each dog
contributes one value per presentation per pairing, so the test is over dogs. Sign-flip
permutation is the natural null here: under H0 the sign of a dog's difference is
arbitrary.

**3b. One mixed model** across all of Part 1, which uses the trial-level data without
pretending the trials are independent.

In [ ]:
def pairing_contrast(df, pairing, a, b, label, n_perm=20000):
    """One dog-level paired contrast with effect size and interval."""
    sub = df[(df.pairing == pairing) & (df.presentation.isin([a, b]))]
    dog_lvl = H.to_dog_level(sub, "investigation_s", ["presentation"])
    wide, dropped = H.paired_wide(dog_lvl, "investigation_s_mean", "presentation")
    if a not in wide or b not in wide:
        return None
    test = H.perm_paired(wide[a], wide[b], n_perm=n_perm)
    hl = H.hodges_lehmann(wide[a], wide[b], n_boot=4000)
    return {"contrast": label, "pairing": pairing, "n_dogs": test["n"],
            "dropped_incomplete": dropped,
            "median_diff_s": round(test["median_diff"], 3),
            "hodges_lehmann_s": round(hl["hodges_lehmann"], 3),
            "ci_low": round(hl["ci_low"], 3), "ci_high": round(hl["ci_high"], 3),
            "rank_biserial": round(H.rank_biserial_paired(wide[a], wide[b]), 3),
            "p_raw": test["p_perm"],
            "mdd_dz_80pct": round(H.min_detectable_d(test["n"]), 3)}


rows = []
for pairing in sorted(part1.pairing.unique()):
    # habituation: first vs third presentation of the repeated odour
    rows.append(pairing_contrast(part1, pairing, "A1", "A3", "habituation:" + pairing))
    # discrimination: fourth (habituated) vs novel, in the dual-box phase
    rows.append(pairing_contrast(part1, pairing, "B", "A4", "discrimination:" + pairing))

part1_contrasts = pd.DataFrame([r for r in rows if r])
part1_contrasts

In [ ]:
# BH-FDR across the pre-declared confirmatory family only
fam = part1_contrasts[part1_contrasts.contrast.isin(CONFIRMATORY)].copy()
adj = H.bh_fdr(fam.p_raw.values, q=0.05)
fam = fam.assign(p_bh=adj.p_bh.values, survives_fdr=adj.reject.values)
fam[["contrast", "n_dogs", "hodges_lehmann_s", "ci_low", "ci_high",
     "rank_biserial", "p_raw", "p_bh", "survives_fdr"]]

In [ ]:
# 3b. Mixed model over all Part 1 trials, dog as random intercept.
# period is included because room and the owner-distraction task are perfectly
# aliased with it - the model cannot separate them and neither can anything else.
lmm = M.fit_part1_lmm(part1)
print(f"dogs (groups): {lmm.model.n_groups}   trials: {lmm.model.nobs:.0f}   converged: {lmm.converged}")
print(f"between-dog SD (log scale): {float(np.sqrt(lmm.cov_re.iloc[0, 0])):.3f}")
lmm.summary().tables[1]

### 3c. The dual-box phase is a preference, not two durations

A4 and B are collected simultaneously under a near-fixed total time budget, so a paired
test on them is structurally a preference test. Modelling B/(A4+B) directly says so, and
the intercept tested against 0.5 is the quantity of interest.

In [ ]:
pref = S.add_preference_index(part1) if data["synthetic"] else None
pref_result = M.fit_preference_index(pref)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in pref_result.items()})
print("\nInterpretation: mean preference for the novel box, 0.5 = no preference.")

## 4. Part 2 — one value per dog, then a model that keeps the bins

### 4a. The ceiling diagnostic

Before re-analysing, confirm the unit the published tests actually used. `h_max` gives
the largest Kruskal-Wallis H attainable for a set of group sizes under perfect
separation. A reported H above the ceiling for 9/9/6 dogs cannot have come from dogs.

In [ ]:
published_H = {"joy behaviours": 23.98, "owner proximity": 23.11,
               "stress behaviours": 20.96}
dog_sizes = [9, 9, 6]
bin_sizes = [12, 12, 12]
ceiling_dogs, ceiling_bins = H.h_max(dog_sizes), H.h_max(bin_sizes)
print(f"H_max for 9/9/6 dogs      : {ceiling_dogs:.2f}")
print(f"H_max for 12/12/12 bins   : {ceiling_bins:.2f}\n")
for name, h in published_H.items():
    verdict = "EXCEEDS dog-level ceiling" if h > ceiling_dogs else "within dog-level ceiling"
    print(f"{name:<20} H = {h:>6.2f}  -> {verdict}")

In [ ]:
# 4b. Dog-level summaries: the primary Part 2 analysis.
MEASURES = ["near_owner", "near_stranger", "stress_behaviour", "joy_behaviour"]

def part2_dog_level(df, measure):
    """Proportion of each dog's own sample points showing the behaviour."""
    g = df.groupby(["dog", "condition", "sex"], observed=True)[measure]
    return g.mean().reset_index().rename(columns={measure: "prop"})


def part2_test(df, measure, n_perm=20000):
    d = part2_dog_level(df, measure)
    groups = [d.loc[d.condition == c, "prop"].values
              for c in ["joy", "stress", "blank"]]
    res = H.perm_kruskal(groups, n_perm=n_perm)
    return {"measure": measure, "n_dogs": sum(res["group_sizes"]),
            "group_sizes": "/".join(map(str, res["group_sizes"])),
            "H": round(res["H"], 3), "H_max": round(res["H_max_possible"], 2),
            "epsilon_sq": round(H.epsilon_squared(groups), 4),
            "p_raw": res["p_perm"],
            "median_joy": round(float(np.median(groups[0])), 3),
            "median_stress": round(float(np.median(groups[1])), 3),
            "median_blank": round(float(np.median(groups[2])), 3)}


part2_results = pd.DataFrame([part2_test(part2, m) for m in MEASURES])
adj2 = H.bh_fdr(part2_results.p_raw.values, q=0.05)
part2_results = part2_results.assign(p_bh=adj2.p_bh.values,
                                     survives_fdr=adj2.reject.values)
part2_results

In [ ]:
# 4c. Binomial GLMM on the bin-level binary data: keeps the within-trial
# information while leaving the dog in the error term.
glmm = M.fit_part2_binomial_glmm(part2, "near_owner")
M.glmm_table(glmm)

In [ ]:
# 4d. The three postures are compositional - one multinomial model, not three tests.
posture_model = M.fit_posture_multinomial(part2)
print(f"log-likelihood {posture_model.llf:.1f}, pseudo-R2 {posture_model.prsquared:.4f}")
print("\nMarginal posture proportions by condition (descriptive only):")
pd.crosstab(part2.condition, part2.posture, normalize="index").round(3)

## 5. Sensitivity: what the design could have detected

The published nulls — no stress-versus-joy difference, no behavioural mirroring — are
reported as findings. At these group sizes they are not informative. This cell states
what each comparison could detect at 80% power, which is the statement that should
accompany every null in the paper.

In [ ]:
sens = pd.DataFrame([
    {"comparison": "Part 2 joy vs stress", "design": "9 vs 9 dogs",
     "min_detectable": H.min_detectable_d(9, 9), "metric": "Cohen's d"},
    {"comparison": "Part 2 joy vs blank", "design": "9 vs 6 dogs",
     "min_detectable": H.min_detectable_d(9, 6), "metric": "Cohen's d"},
    {"comparison": "Part 2 stress vs blank", "design": "9 vs 6 dogs",
     "min_detectable": H.min_detectable_d(9, 6), "metric": "Cohen's d"},
    {"comparison": "Part 1 pairing-level paired", "design": "40 dogs",
     "min_detectable": H.min_detectable_d(40), "metric": "d_z"},
    {"comparison": "Part 1 direction-specific", "design": "~20 dogs",
     "min_detectable": H.min_detectable_d(20), "metric": "d_z"},
]).round(3)
sens

## 6. Coder reliability

Neither primary outcome is machine-derived, and no reliability statistic appears in the
paper. These are the two numbers a revision needs, computed on a blind double-coded
subsample of at least 20% of trials, stratified across conditions and collection periods.
ICC for the durations (not Pearson r, which is blind to systematic duration offsets) and
kappa per ethogram item (not percentage agreement, which low base rates inflate).

In [ ]:
rel = S.simulate_reliability()
icc = H.icc21(rel[["duration_coder_a", "duration_coder_b"]].values)
kappa = H.cohen_kappa(rel.lipslick_coder_a, rel.lipslick_coder_b)
pearson = st.pearsonr(rel.duration_coder_a, rel.duration_coder_b).statistic
pct_agree = float((rel.lipslick_coder_a == rel.lipslick_coder_b).mean())
print(f"investigation duration  ICC(2,1) = {icc:.3f}   (Pearson r = {pearson:.3f})")
print(f"lip/nose licking        kappa    = {kappa:.3f}   (raw agreement = {pct_agree:.3f})")
print("\nThe gap between each pair is the reason the first statistic is the right one.")

## 7. Reporting

One table of every test run, with its unit of analysis, effect size, interval and
adjusted p — so the test count is auditable rather than reconstructed from the prose.

In [ ]:
report = pd.concat([
    fam.assign(part="Part 1", unit="dog", family="confirmatory")
       [["part", "contrast", "unit", "family", "n_dogs", "hodges_lehmann_s",
         "ci_low", "ci_high", "p_raw", "p_bh", "survives_fdr"]]
       .rename(columns={"contrast": "test", "hodges_lehmann_s": "effect"}),
    part2_results.assign(part="Part 2", unit="dog", family="primary")
       [["part", "measure", "unit", "family", "n_dogs", "epsilon_sq",
         "p_raw", "p_bh", "survives_fdr"]]
       .rename(columns={"measure": "test", "epsilon_sq": "effect"}),
], ignore_index=True)
report.to_csv(OUT / "test_register.csv", index=False)
print(f"{len(report)} tests registered -> outputs/test_register.csv")
report

In [ ]:
# Dog-level presentation: per-dog points over medians, replacing mean +- SEM bars
# on skewed, floor-bounded data.
dl = part2_dog_level(part2, "near_owner")
fig, ax = plt.subplots(figsize=(5.2, 4))
order = ["blank", "joy", "stress"]
sns.stripplot(data=dl, x="condition", y="prop", order=order, ax=ax,
              size=7, jitter=0.18, alpha=0.8, color="#4C72B0")
meds = dl.groupby("condition").prop.median().reindex(order)
ax.hlines(meds.values, np.arange(3) - 0.28, np.arange(3) + 0.28,
          color="black", lw=2, zorder=5)
ax.set_ylabel("proportion of sample points near owner")
ax.set_xlabel("odour condition")
ax.set_title("One point per dog, bar = median  [SYNTHETIC DATA]")
sns.despine(ax=ax)
fig.tight_layout(); fig.savefig(OUT / "part2_dog_level.png", dpi=200, bbox_inches="tight")

## 8. What this notebook refuses to do

```
donor random effect     -> not estimable: 1 donor. A species-level claim has no
                           error term in this design. Add donors, or retitle to
                           "body odours of a conspecific collected in different
                           contexts".
room / distraction task -> not separable from collection period (14+17 = 31 = room 1;
                           12 = room 2; 17+12 = 29 = the distraction group). The
                           single period covariate cannot clear any of them.
exertion vs emotion     -> not separable: the joy sample is the only one preceded by
                           two minutes of exercise, and every significant
                           discrimination contrast involves it. Needs an
                           arousal-matched control sample, or GC-MS.
```

Each of these is a design change, not an analysis choice, and stating them as such is
part of the output.